# Part 5: The Analyst Report

After you have successfully deployed your pipeline and run the **Burst** profile (500 messages) in the test apparatus, you need to extract the results and answer a few questions.

We use `boto3` to scan the DynamoDB table, handling pagination automatically, and convert the results into standard Python dictionaries and floats.

## Setup: Configure Your Student ID
Replace `YOURID` below with the exact student ID you used for deployment.

In [ ]:
%pip install boto3 pandas
STUDENT_ID = "17862"  # <--- Change this
TABLE_NAME = f"adflow-{STUDENT_ID}-results"
REGION = "us-east-1"
print(f"Target Table: {TABLE_NAME}")

## Step 1: Export Data from DynamoDB
This cell connects to your DynamoDB table, downloads all records, and converts the Decimal values back to standard floats.

In [ ]:
import boto3
import pandas as pd
from decimal import Decimal
from collections import Counter

# Note: This uses your active AWS credentials (from `aws configure` or exported environment variables)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

results = []
response = table.scan()
results.extend(response.get("Items", []))

# Handle pagination if the table has more than 1 MB of data
while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    results.extend(response.get("Items", []))

print(f"\nLoaded {len(results)} records from DynamoDB.")

# Convert Decimal types to Python floats for easier math/plotting
for item in results:
    for key in ["winning_bid_amount", "winning_score", "score_margin"]:
        if key in item and isinstance(item[key], Decimal):
            item[key] = float(item[key])

if results:
    df = pd.DataFrame(results)
    print(f"\nCreated DataFrame with shape: {df.shape}")
    print("\nSample record (first row):\n")
    print(df.head(1))


## Section 1: Pipeline Evidence
Print the total records and a quick count of auction wins per advertiser across the entire dataset to prove your pipeline successfully routed messages.

In [ ]:
# Print the total number of records
print(f"Total pipeline records: {len(results)}")

# Compute and print the auction wins per advertiser (overall)
print("\nOverall Winners:")
print(df['winning_advertiser_id'].value_counts())


**Evidence Requirement:** Don't forget to push a screenshot of the **Test Apparatus** (showing a completed Burst run) to a `screenshots/` directory in this repo when submitting.

---
## Q1: Results Analysis

**Question:** Which advertiser won the most auctions overall? Which advertiser won the most in the `sports` content category specifically? Why do the overall and sports-specific rankings differ? Explain in 2â€“3 sentences, referencing the relevance multiplier table.

In [ ]:
# Find the top winner in the 'sports' category
sports_df = df[df['content_category'] == 'sports']
print(f"Sports records: {len(sports_df)}")
print("\nSports Winners:")
print(sports_df['winning_advertiser_id'].value_counts())


**Your Answer (Q1):**

When I looked at the data, the overall winners were just the advertisers with the biggest bids. This makes sense because they have the most capital. But in the sports category, sportswear started winning a lot more. This happened because of the 1.4x relevance multiplier we put in the code. Even if a sportswear company bid 4.00 and a fast food company bid 5.00, the multiplier pushed the sportswear score to 5.6. This made it the winner. It was cool to see the math actually work to favor relevant ads over just high bidders.

---
## Q2: Code Reflection

Answer **one** of the following (your choice):
 
* **Option A (Scale & Limits):** The test apparatus sent messages in small batches. If traffic suddenly spiked from 10 opportunities a second to 10,000 a second, what specific components of our current pipeline (SQS limits, Lambda concurrency, DynamoDB throughput) would become bottlenecks first, and what AWS settings would you adjust to handle the load?
* **Option B (The Distributed Process):** Writing code for an event-driven, queue-based pipeline is very different from writing a single local script. What was the most challenging part of getting SQS, Lambda, and DynamoDB to communicate correctly, or the most confusing bug you encountered, and what did it teach you about distributed architecture?

A well-argued two-paragraph response is sufficient for either option.

**Your Answer (Q2):**

This was my first time ever cloning a project from a professor and trying to sync it to my own GitHub account. Honestly, just getting the terminal to talk to my account was a huge hurdle. I spent a lot of time just figuring out how to push my code to the right 17862 branch. The most frustrating technical bug was the data types. I did not realize that SQS and DynamoDB handle numbers so differently. My code kept crashing because I was trying to send normal Python floats to a database that requires Decimals. Fixing that taught me that in a distributed system, you cannot just assume things will work like they do in a local script. You have to be super careful about how data moves between different services.

Once I got the pipeline actually running, the latency was really high at around 5 seconds. I had to learn how to tune the performance. We realized that by moving the database connection to the very top of the script in the global scope, the Lambda function did not have to reconnect every single time it woke up. We also bumped the memory up to 1024MB. I learned this gives the Lambda a faster CPU for free. Seeing the latency drop from a slow red bar to a green 389ms was the best part of the project. It taught me that building a cloud app is not just about writing the logic. It is about understanding the infrastructure and how to make the hardware work for your code.